In [ ]:
import spacy, torch
from datasets import concatenate_datasets, load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM, logging
from nltk.corpus import wordnet
from nltk.wsd import lesk
# ERROR With dataset creation: train-augmented-hybrid does not have train-original data

In [2]:
original_dataset = load_dataset('glue', 'mrpc')
nlp = spacy.load("en_core_web_sm") # NER algorithm

#### Utility Functions

In [3]:
# replace the first instance of a word with another word, but only after a certain point
def replace_at(sentence, target, newword, idx):
    # split the sentence at idx
    # leave the part of the sentence before idx untouched
    # for the part after idx, replace the first instance of target with newword
    return sentence[:idx] + sentence[idx:].replace(target, newword, 1)

In [4]:
# get a set of synonyms for a given word via wordnet
def query_wordnet(word, _):
    syns = set()
    # pos=n retrieves only nouns (pos = Part-of-Speech)
    for synset in wordnet.synsets(word, pos='n'):
        for lemma in synset.lemmas():
            syns.add(lemma.name().replace('_',' ').lower())
    if word in syns: syns.remove(word)
    return syns

In [5]:
# get a set of synonyms for a given word via the lesk algorithm
def query_lesk(word, sen):
    syns = set()
    # pos=n retrieves only nouns (pos = Part-of-Speech)
    synset = lesk(sen.split(' '), word, pos='n')
    if not synset: return syns
    for lemma in synset.lemmas():
        syns.add(lemma.name().replace('_',' ').lower())
    if word in syns: syns.remove(word)
    return syns

In [6]:
# get a set of synonyms for each noun in each sentence
# each noun is stored in its own dict object
def get_synonyms(sentences, query_fn=None):
    # synsets = [{
    # 	'sen-idx': int, position of sentence in sentences list
    # 	'noun': str, noun to be replaced,
    #   'noun-idx': int, index in sentence where noun starts (incase sentence has multiple of the same noun)
    # 	'syns': set[str] (optional), synonyms of noun obtained from wordnet/lesk
    # }]
    synsets = []
    docs = nlp.pipe(sentences)

    # nlp pipe makes a document for each sentence
    # document assigns PoS tag to each word in the sentence, stored as tokens
    # tokens contain original text data, PoS tag, and other things
    for idx, (sen, doc) in enumerate(zip(sentences, docs)):
        for token in doc:
            # skip token if it is not a noun
            if token.pos_ != 'NOUN': continue
            # find synonyms if query function specified, otherwise just find nouns
            if query_fn:
                synsets.append({
                    'sen-idx': idx,
                    'noun': token.text,
                    'noun-idx': token.idx,
                    'syns': query_fn(token.text, sen)
                })
            else:
                synsets.append({
                    'sen-idx': idx,
                    'noun': token.text,
                    'noun-idx': token.idx
                })
    return synsets

In [7]:
# return prediction scores for each noun in each sentence
# prediction scores are logit probabilities over the entire vocabulary
def get_predictions_bert(sentences, synsets, model, tokenizer):
    # generate a masked sentence for each noun in each sentence
    masked_sentences = []
    for synset in synsets:
        i = synset['sen-idx']
        noun = synset['noun']
        ni = synset['noun-idx']
        masked_sentences.append(replace_at(sentences[i], noun, '[MASK]', ni))
    
    inputs = tokenizer(masked_sentences, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        # tensor of size batch size x len(longest encoded sentence) x vocab size
        logits = model(**inputs).logits.cpu()
    # input ids is a tensor of batch size x len(longest encoded sentence)
    # torch.where returns a copy of tensor with mask tokens set to 1 and all other values set to 0
    # argmax returns index of mask token for each row/sentence as a tensor of length batch size
    mask_locations = torch.where(inputs.input_ids == tokenizer.mask_token_id, 1, 0).argmax(dim=1).cpu()
    mask_preds = []
    for row, mask_idx in zip(logits, mask_locations):
        mask_preds.append(row[mask_idx])
    return mask_preds

In [8]:
# return prediction scores for each noun in each sentence
# prediction scores are logit probabilities over the entire vocabulary
def get_predictions_gpt2(sentences, synsets, model, tokenizer):
    # generate prompts to ask gpt2 what the missing word is
    prompts = []
    for synset in synsets:
        i = synset['sen-idx']
        noun = synset['noun']
        ni = synset['noun-idx']
        sen = sentences[i]

        sen_before = sen[:sen.find(noun, ni)]
        sen_after = sen[sen.find(noun, ni) + len(noun):]
        prompts.append(f"Fill in the missing word.\nSentence: {sen_before}___{sen_after}\nAnswer:")
    
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        # tensor of size batch size x len(longest encoded sentence) x vocab size
        logits = model(**inputs).logits.cpu()
    # since padding is left, all sentences end at the last token
    last_token_index = logits.shape[1] - 1
    return [row[last_token_index] for row in logits]

#### Augmentation Functions

In [9]:
# convert sentence columns into a single prompt column
# final transformation applied to all datasets
def aug(batch):
    prompts = []
    for sen1, sen2 in zip(batch['sentence1'], batch['sentence2']):
        prompts.append(f"Sentence1: {sen1}\nSentence2: {sen2}\nDo these sentences mean the same thing? Respond with 1 if they do, or 0 if they don't.")
    return {'text': prompts}

In [10]:
# naive augmentation which replaces nouns with any synonym
def aug_naive(batch, query_fn):
    for col in ['sentence1', 'sentence2']:
        synsets = get_synonyms(batch[col], query_fn)

        for synset in synsets:
            i = synset['sen-idx']
            noun = synset['noun']
            ni = synset['noun-idx']
            # for loop followed by immediate break just gets any one value from set
            for syn in synset['syns']:
                batch[col][i] = replace_at(batch[col][i], noun, syn, ni)
                break
    return aug(batch)

In [11]:
def aug_llm(batch, model, tokenizer, pred_fn):
    for col in ['sentence1', 'sentence2']:
        # use get_synonyms function to get nouns. retrieved synonyms are discarded
        synsets = get_synonyms(batch[col])
        preds = pred_fn(batch[col], synsets, model, tokenizer)

        for synset, pred in zip(synsets, preds):
            i = synset['sen-idx']
            noun = synset['noun']
            ni = synset['noun-idx']

            best_token = pred.argmax()
            best_word = tokenizer.decode(best_token)
            batch[col][i] = replace_at(batch[col][i], noun, best_word, ni)
    return aug(batch)

In [12]:
def aug_hybrid(batch, model, tokenizer, pred_fn):
    is_bert = 'bert' in model.name_or_path[0]
    
    for col in ['sentence1', 'sentence2']:
        synsets = get_synonyms(batch[col], query_lesk)
        preds = pred_fn(batch[col], synsets, model, tokenizer)

        for synset, pred in zip(synsets, preds):
            # find synonym from synset with the highest prediction score
            best_syn = synset['noun'] # default to original noun if no synonym found
            best_score = 0
            for syn in synset['syns']:
                # with BERT tokenizer, first element is always start sentence token
                encoded_syn = tokenizer.encode(syn)[1 if is_bert else 0] 
                score = pred[encoded_syn].item()
                if score > best_score:
                    best_syn = syn
                    best_score = score
            # replace
            i = synset['sen-idx']
            batch[col][i] = replace_at(batch[col][i], synset['noun'], best_syn, synset['noun-idx'])
    return aug(batch)

#### Create Datasets

In [13]:
new_datasets = []

In [14]:
# convert original dataset into dataset of prompts
new_datasets.append(
    original_dataset['train'].map(
        aug,
        batched=True,
        batch_size=16,
        remove_columns=['sentence1','sentence2','idx']
    )
)

In [15]:
# apply naive augmentations and conver to prompts
for query_fn in [query_wordnet, query_lesk]:
    new_datasets.append(
        original_dataset['train'].map(
            lambda batch: aug_naive(batch, query_fn),
            batched=True,
            batch_size=16,
            remove_columns=['sentence1','sentence2','idx']
        )
    )

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

In [16]:
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased", device_map='cuda')
model.eval()
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased", device_map='cuda')

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [17]:
for aug_fn in [aug_llm, aug_hybrid]:
    new_datasets.append(
        original_dataset['train'].map(
            lambda batch: aug_fn(batch, model, tokenizer, get_predictions_bert),
            batched=True,
            batch_size=16,
            remove_columns=['sentence1','sentence2','idx']
        )
    )

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

In [18]:
# switch model from bert to gpt2 and rerun llm augmentation functions
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", device_map='cuda')
model.eval()
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2", device_map='cuda')

tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

In [19]:
for aug_fn in [aug_llm, aug_hybrid]:
    new_datasets.append(
        original_dataset['train'].map(
            lambda batch: aug_fn(batch, model, tokenizer, get_predictions_gpt2),
            batched=True,
            batch_size=16,
            remove_columns=['sentence1','sentence2','idx']
        )
    )

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

In [20]:
# add original data to the augmented datasets
# remove duplicate rows
for i in range(1, len(new_datasets)):
    df = concatenate_datasets([new_datasets[0], new_datasets[i]]).to_pandas()
    df = df.drop_duplicates().reset_index(drop=True)
    new_datasets[i] = Dataset.from_pandas(df)

In [21]:
val = original_dataset['validation'].map(
    aug,
    batched=True,
    remove_columns=['sentence1','sentence2','idx']
)

test = original_dataset['test'].map(
    aug,
    batched=True,
    remove_columns=['sentence1','sentence2','idx']
)

In [22]:
DatasetDict({
    'train-original':           new_datasets[0],
    'train-augmented-wordnet':  new_datasets[1],
    'train-augmented-lesk':     new_datasets[2],
    'train-augmented-bert':     new_datasets[3],
    'train-augmented-hybert':   new_datasets[4],
    'train-augmented-gpt2':     new_datasets[5],
    'train-augmented-hygpt2':   new_datasets[6],
    'validation':               val,
    'test':                     test
}).save_to_disk('dataset.hf')

Saving the dataset (0/1 shards):   0%|          | 0/3668 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7317 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7292 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7247 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3685 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7325 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6358 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1725 [00:00<?, ? examples/s]